# **7일차 실습: 도메인 특화 도구 만들기 및 테스트**

## 학습 목표
1. LangChain Tool 개념 이해
2. AI를 활용한 도구 자동 생성
3. 생성한 도구 테스트
4. 도메인 특화 에이전트에 적용

## 실습 단계
1. **도구 설계**: 팀 프로젝트에 필요한 도구 정의
2. **도구 생성**: AI를 활용하여 도구 코드 자동 생성
3. **도구 테스트**: 생성된 도구가 올바르게 동작하는지 확인
4. **에이전트 적용**: domain-agent에 통합 (다음 단계)

---

## 1. AI 도구 생성 프롬프트 확인

팀 프로젝트에 필요한 도구를 AI로 자동 생성하기 위한 프롬프트입니다.

**프롬프트 파일 위치**: `../../make_tool_prompt.txt`

In [1]:
# 프롬프트 파일 읽기
with open("../make_tool_prompt.txt", "r", encoding="utf-8") as f:
    prompt_template = f.read()

print("=" * 80)
print("AI 도구 생성 프롬프트")
print("=" * 80)
print(prompt_template)
print("\n✓ 프롬프트 로드 완료")
print("\n💡 이 프롬프트를 ChatGPT, Claude 등에 복사하여 사용하세요!")

AI 도구 생성 프롬프트
당신은 LangChain/LangGraph 기반 AI Agent를 위한 Python Tool 개발 전문가입니다.

아래 **[사용자 요구사항]** 에 작성된 내용을 바탕으로 LangChain Tool을 생성하세요.

---

# 사용자 요구사항

## 여기에 원하는 기능을 작성하세요.

<이곳에 원하는 Tool 기능을 작성>

---

# 구현 규칙

다음 규칙을 반드시 지키세요.

## 출력 형식

* Python 코드만 출력합니다.
* 코드 외의 설명은 출력하지 않습니다.
* `from langchain_core.tools import tool`을 사용합니다.
* `@tool(parse_docstring=True)` 데코레이터를 사용합니다.
* Python 3.11 이상 기준으로 작성합니다.

## 함수 작성 규칙

* 함수명은 기능에 맞게 작성합니다.
* 모든 매개변수에는 타입 힌트를 작성합니다.
* 반환 타입도 작성합니다.
* 필요한 import는 함수 내부에서 수행합니다.
* 가능한 표준 라이브러리를 우선 사용합니다.
* 외부 라이브러리가 필요한 경우 import도 함께 작성합니다.

## Docstring

반드시 Google Style Docstring을 작성합니다.

예시 형식

```python
"""도구 설명

Args:
    parameter1: 설명
    parameter2: 설명

Returns:
    반환값 설명
"""
```

## 예외 처리

반드시 예외 처리를 구현합니다.

형식

```python
try:
    ...
    return "성공 메시지"
except Exception as e:
    return f"실패: {str(e)}"
```

## 테스트 코드

마지막에 아래 코드를 추가합니다.

```python
print(f"도구 이름: {함수명.name}")
print(f"도구 설명: {함수명.description}")
```

## 코드 품질

* 읽기 쉬운 코드로 작성합니다.
* 

## 2. 도구 설계 가이드

### 좋은 도구의 조건

1. **단일 책임**: 하나의 명확한 기능만 수행
2. **명확한 입력/출력**: 매개변수와 반환값이 명확
3. **에러 처리**: 예외 상황을 적절히 처리
4. **좋은 설명**: Docstring으로 도구의 기능을 명확히 설명

### 도메인별 도구 예시

**쇼핑 도메인:**
- 상품 검색
- 가격 비교
- 재고 확인
- 리뷰 조회

**법령 도메인:**
- 법령 검색
- 조문 조회
- 판례 검색
- 법령 해석

**의료 도메인:**
- 증상 검색
- 병원 찾기
- 약 정보 조회
- 건강 정보 제공

**여행 도메인:**
- 항공권 검색
- 호텔 검색
- 관광지 정보
- 날씨 확인

---

## 3. 도구 생성 프로세스

### Step 1: 팀 프로젝트 도메인 및 필요한 도구 정의

**TODO: 팀에서 선택한 도메인과 필요한 도구를 작성하세요**

```
팀 도메인: 오픈 LLM 실행 및 성능 비교·평가

필요한 도구 목록:

1. 도구명: `search_openrouter_llms`
   - 입력: 검색어, 지원 언어, 비교할 모델 수
   - 출력: 무료 오픈 LLM의 모델 ID, 모델명, 개발사, 설명, 컨텍스트 길이, 가격, 지원 매개변수
   - 역할: OpenRouter API에서 무료로 실행 가능한 텍스트 LLM을 검색하고, 서로 다른 개발사의 비교 후보를 선택

2. 도구명: `run_openrouter_battle`
   - 입력: 후보 모델 ID 목록, 사용자 질문, 최대 출력 토큰 수, temperature
   - 출력: 모델별 실제 답변, 응답 시간, 실행 성공 여부, 오류 메시지
   - 역할: 선택된 OpenRouter 무료 모델에 동일한 질문과 생성 조건을 전달하여 실제 답변과 실행 성능을 측정

3. 도구명: `evaluate_llm_responses`
   - 입력: 사용자 질문, 모델별 실제 답변 및 실행 결과
   - 출력: 모델별 관련성, 완성도, 명확성, 실용성, 응답 속도, 실행 안정성 점수, 장단점, 추천 용도 및 최종 순위
   - 역할: 동일한 평가 기준으로 모델별 답변과 성능을 비교하고, 사용자 목적에 가장 적합한 모델과 선정 이유를 제시

4. 도구명: `generate_report_markdown`
   - 입력: 사용자 질문, 모델 실행 결과 JSON, 비교 평가 결과 JSON, Markdown 파일명
   - 출력: 생성된 Markdown 보고서의 절대 파일 경로
   - 역할: 모델별 전체 답변, 응답 시간, 세부 평가 점수, 장단점, 추천 용도, 최종 순위를 Markdown 보고서로 생성
```

### Step 2: AI로 도구 생성하기

**사용 방법:**

1. 위의 프롬프트 템플릿을 복사
2. `<이곳에 원하는 Tool 기능을 작성>` 부분에 팀의 도구 요구사항 작성
3. ChatGPT, Claude 등에 입력하여 코드 생성
4. 생성된 코드를 아래 셀에 붙여넣기

**예시 입력:**
```
쇼핑 도메인의 상품 검색 도구를 만들어주세요.

기능:
- 상품명으로 검색
- 가격 범위 필터링
- 카테고리 필터링
- 검색 결과를 JSON 형태로 반환
```

---

## 4. 생성된 도구 코드 테스트

**TODO: AI가 생성한 도구 코드를 아래에 붙여넣으세요**

**중요:** 
- 코드를 실행하기 전에 반드시 검토하세요
- 필요한 외부 라이브러리가 있다면 먼저 설치하세요
- 실제 API 키가 필요한 경우 .env 파일에 추가하세요

In [ ]:
"""Open LLM Battle Agent custom tools."""

import json
import os
import re
import time
from pathlib import Path

import requests
from dotenv import load_dotenv
from langchain_core.tools import tool

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_API_URL = "https://openrouter.ai/api/v1"


def _headers() -> dict:
    if not OPENROUTER_API_KEY:
        raise ValueError(".env에서 OPENROUTER_API_KEY를 찾을 수 없습니다.")

    return {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
        "X-Title": "Open LLM Battle Agent",
    }


def _json(value: dict) -> str:
    return json.dumps(
        value,
        ensure_ascii=False,
        indent=2,
        default=str,
    )


@tool(parse_docstring=True)
def search_openrouter_llms(
    search_query: str,
    language: str = "multilingual",
    model_count: int = 3,
) -> str:
    """OpenRouter에서 현재 무료로 실행 가능한 오픈 LLM을 검색합니다.

    Args:
        search_query: 모델명 또는 개발사 검색어입니다. 예: Qwen, Gemma.
        language: 원하는 지원 언어 조건입니다.
        model_count: 반환할 모델 수이며 1 이상 5 이하입니다.

    Returns:
        모델 ID와 Provider 및 모델 정보를 JSON으로 반환합니다.
    """
    try:
        if not search_query.strip():
            return "실패: 검색어가 비어 있습니다."

        if not 1 <= model_count <= 5:
            return "실패: model_count는 1 이상 5 이하이어야 합니다."

        response = requests.get(
            f"{OPENROUTER_API_URL}/models",
            headers=_headers(),
            timeout=30,
        )
        response.raise_for_status()

        router_models = response.json().get("data", [])

        free_models = [
            model
            for model in router_models
            if float((model.get("pricing") or {}).get("prompt", 1)) == 0
            and float((model.get("pricing") or {}).get("completion", 1)) == 0
            and "text"
            in (model.get("architecture") or {}).get(
                "output_modalities",
                [],
            )
            and model.get("id") != "openrouter/free"
        ]

        searchable_families = (
            "gemma",
            "minimax",
            "nemotron",
            "glm",
            "liquid",
            "inkling",
            "laguna",
            "cohere",
            "qwen",
            "llama",
            "mistral",
            "deepseek",
            "phi",
            "gpt-oss",
        )

        query_words = set(
            re.findall(
                r"[a-z0-9.-]+",
                search_query.lower(),
            )
        )

        requested_families = [
            family
            for family in searchable_families
            if family in query_words
        ]

        # 특정 모델 계열이 명시된 경우 해당 계열을 검색합니다.
        if requested_families:
            candidates = [
                model
                for model in free_models
                if any(
                    family
                    in (
                        f"{model.get('id', '')} "
                        f"{model.get('owned_by', '')}"
                    ).lower()
                    for family in requested_families
                )
            ]

        # 일반적인 검색에서는 무료 모델 전체를 후보로 사용합니다.
        else:
            candidates = free_models

        # 같은 개발사의 모델이 중복되지 않도록 선택합니다.
        if not requested_families:
            diverse_candidates = []
            seen_owners = set()

            for model in candidates:
                owner = str(
                    model.get("owned_by")
                    or model.get("id", "").split("/", 1)[0]
                )

                if owner.lower() in seen_owners:
                    continue

                seen_owners.add(owner.lower())
                diverse_candidates.append(model)

            candidates = diverse_candidates

        if not candidates:
            return _json(
                {
                    "status": "failed",
                    "message": (
                        "현재 실행 가능한 조건 일치 모델을 "
                        "찾지 못했습니다."
                    ),
                    "search_query": search_query,
                }
            )

        models = []

        for candidate in candidates[:model_count]:
            model_id = candidate["id"]

            models.append(
                {
                    "model_id": model_id,
                    "name": candidate.get("name"),
                    "owned_by": candidate.get(
                        "id",
                        "",
                    ).split("/", 1)[0],
                    "language_condition": language,
                    "description": candidate.get("description"),
                    "context_length": candidate.get(
                        "context_length"
                    ),
                    "pricing": candidate.get("pricing"),
                    "supported_parameters": candidate.get(
                        "supported_parameters",
                        [],
                    ),
                    "model_url": (
                        f"https://openrouter.ai/{model_id}"
                    ),
                }
            )

        return _json(
            {
                "status": "success",
                "search_query": search_query,
                "language": language,
                "model_count": len(models),
                "models": models,
            }
        )

    except requests.RequestException as error:
        return (
            f"실패: OpenRouter 모델 검색 오류 - {error}"
        )

    except Exception as error:
        return f"실패: {error}"


@tool(parse_docstring=True)
def run_openrouter_battle(
    model_ids: list[str],
    question: str,
    max_tokens: int = 256,
    temperature: float = 0.2,
) -> str:
    """여러 오픈 LLM에 같은 질문을 전달하여 답변과 실행 성능을 측정합니다.

    Args:
        model_ids: 비교할 OpenRouter 무료 모델 ID 목록입니다.
        question: 모든 모델에 동일하게 전달할 질문입니다.
        max_tokens: 최대 출력 토큰 수이며 16 이상 1024 이하입니다.
        temperature: 답변 무작위성 값이며 0 이상 2 이하입니다.

    Returns:
        모델별 답변, 응답 시간, 성공 여부와 오류를 JSON으로 반환합니다.
    """
    try:
        if not model_ids:
            return "실패: 배틀에 사용할 모델 ID가 없습니다."

        if len(model_ids) > 5:
            return "실패: 한 번에 비교할 모델은 최대 5개입니다."

        if not question.strip():
            return "실패: 사용자 질문이 비어 있습니다."

        if not 16 <= max_tokens <= 1024:
            return (
                "실패: max_tokens는 "
                "16 이상 1024 이하이어야 합니다."
            )

        if not 0 <= temperature <= 2:
            return (
                "실패: temperature는 "
                "0 이상 2 이하이어야 합니다."
            )

        results = []

        for model_id in model_ids:
            started = time.perf_counter()

            try:
                response = requests.post(
                    f"{OPENROUTER_API_URL}/chat/completions",
                    headers=_headers(),
                    timeout=90,
                    json={
                        "model": model_id,
                        "messages": [
                            {
                                "role": "user",
                                "content": question,
                            }
                        ],
                        "max_tokens": max_tokens,
                        "temperature": temperature,
                    },
                )

                elapsed = round(
                    time.perf_counter() - started,
                    3,
                )

                if not response.ok:
                    results.append(
                        {
                            "model_id": model_id,
                            "status": "failed",
                            "answer": None,
                            "response_time_seconds": elapsed,
                            "error": response.text[:500],
                        }
                    )
                    continue

                data = response.json()
                choices = data.get("choices") or []

                if not choices:
                    results.append(
                        {
                            "model_id": model_id,
                            "status": "failed",
                            "answer": None,
                            "response_time_seconds": elapsed,
                            "error": "생성된 답변이 없습니다.",
                        }
                    )
                    continue

                results.append(
                    {
                        "model_id": model_id,
                        "status": "success",
                        "answer": (
                            choices[0].get("message") or {}
                        ).get("content"),
                        "response_time_seconds": elapsed,
                        "finish_reason": choices[0].get(
                            "finish_reason"
                        ),
                        "usage": data.get("usage") or {},
                    }
                )

            except requests.RequestException as error:
                results.append(
                    {
                        "model_id": model_id,
                        "status": "failed",
                        "answer": None,
                        "response_time_seconds": round(
                            time.perf_counter() - started,
                            3,
                        ),
                        "error": str(error),
                    }
                )

        success_count = sum(
            item["status"] == "success"
            for item in results
        )

        return _json(
            {
                "status": (
                    "success"
                    if success_count
                    else "failed"
                ),
                "question": question,
                "generation_options": {
                    "max_tokens": max_tokens,
                    "temperature": temperature,
                },
                "success_count": success_count,
                "failure_count": (
                    len(results) - success_count
                ),
                "results": results,
            }
        )

    except Exception as error:
        return f"실패: {error}"


@tool(parse_docstring=True)
def evaluate_llm_responses(
    question: str,
    battle_results_json: str,
) -> str:
    """모델 답변을 정량 기준으로 비교하고 순위를 계산합니다.

    Args:
        question: 모델들에게 전달한 원본 질문입니다.
        battle_results_json: run_openrouter_battle이 반환한 JSON 문자열입니다.

    Returns:
        관련성, 완성도, 명확성, 속도, 안정성 점수와 순위를 JSON으로 반환합니다.
    """
    try:
        results = json.loads(
            battle_results_json
        ).get("results") or []

        successful = [
            result
            for result in results
            if result.get("status") == "success"
            and result.get("answer")
        ]

        if not results:
            return (
                "실패: 평가할 모델 실행 결과가 없습니다."
            )

        if not successful:
            return (
                "실패: 정상적으로 답변을 생성한 "
                "모델이 없습니다."
            )

        fastest = min(
            result["response_time_seconds"]
            for result in successful
        )

        keywords = set(
            re.findall(
                r"[A-Za-z0-9가-힣]{2,}",
                question.lower(),
            )
        )

        evaluations = []

        for result in results:
            if (
                result.get("status") != "success"
                or not result.get("answer")
            ):
                evaluations.append(
                    {
                        "model_id": result.get("model_id"),
                        "status": "failed",
                        "total_score": 0,
                        "reason": result.get(
                            "error",
                            "실행 실패",
                        ),
                    }
                )
                continue

            answer = str(result["answer"])

            matched = [
                word
                for word in keywords
                if word in answer.lower()
            ]

            relevance = (
                round(
                    len(matched)
                    / len(keywords)
                    * 30,
                    2,
                )
                if keywords
                else 15
            )

            completeness = round(
                min(len(answer) / 500, 1) * 20,
                2,
            )

            sentences = [
                sentence.strip()
                for sentence in re.split(
                    r"[.!?。！？\n]+",
                    answer,
                )
                if sentence.strip()
            ]

            average = (
                sum(map(len, sentences))
                / len(sentences)
                if sentences
                else len(answer)
            )

            if 15 <= average <= 120:
                clarity = 20
            elif average <= 180:
                clarity = 14
            else:
                clarity = 8

            speed = round(
                min(
                    fastest
                    / max(
                        result[
                            "response_time_seconds"
                        ],
                        0.001,
                    ),
                    1,
                )
                * 20,
                2,
            )

            total = round(
                relevance
                + completeness
                + clarity
                + speed
                + 10,
                2,
            )

            evaluations.append(
                {
                    "model_id": result.get("model_id"),
                    "status": "success",
                    "total_score": total,
                    "scores": {
                        "relevance": relevance,
                        "completeness": completeness,
                        "clarity": clarity,
                        "speed": speed,
                        "stability": 10,
                    },
                    "response_time_seconds": result[
                        "response_time_seconds"
                    ],
                    "answer_length": len(answer),
                    "matched_keywords": matched,
                }
            )

        ranking = sorted(
            evaluations,
            key=lambda item: item.get(
                "total_score",
                0,
            ),
            reverse=True,
        )

        for index, item in enumerate(
            ranking,
            start=1,
        ):
            item["rank"] = index

        winner = ranking[0]

        return _json(
            {
                "status": "success",
                "question": question,
                "evaluation_method": {
                    "relevance": 30,
                    "completeness": 20,
                    "clarity": 20,
                    "speed": 20,
                    "stability": 10,
                },
                "ranking": ranking,
                "winner": {
                    "model_id": winner["model_id"],
                    "total_score": winner[
                        "total_score"
                    ],
                    "reason": (
                        "정량 평가 점수가 가장 높은 "
                        "모델입니다."
                    ),
                },
                "important_notice": (
                    "정답 데이터가 없으므로 "
                    "사실 정확성은 자동 확정할 수 없습니다."
                ),
            }
        )

    except json.JSONDecodeError:
        return (
            "실패: battle_results_json이 "
            "올바른 JSON 형식이 아닙니다."
        )

    except Exception as error:
        return f"실패: {error}"


@tool(parse_docstring=True)
def generate_report_markdown(
    battle_results_json: str,
    evaluation_results_json: str,
    report_filename: str = "open_llm_battle_report.md",
) -> str:
    """배틀 결과와 비교 평가를 Markdown 보고서로 생성합니다.

    Args:
        battle_results_json: run_openrouter_battle이 반환한 JSON 문자열입니다.
        evaluation_results_json: evaluate_llm_responses가 반환한 JSON 문자열입니다.
        report_filename: 경로를 제외한 Markdown 파일명입니다.

    Returns:
        생성된 Markdown 보고서의 절대 경로 또는 오류 메시지를 반환합니다.
    """
    try:
        battle = json.loads(
            battle_results_json
        )
        evaluation = json.loads(
            evaluation_results_json
        )

        filename = Path(
            report_filename
        ).name

        if not filename.lower().endswith(".md"):
            filename += ".md"

        # 실행 위치의 outputs 디렉터리에 저장합니다.
        output_dir = Path.cwd() / "outputs"
        output_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        output_path = output_dir / filename

        question = str(
            battle.get(
                "question",
                "-",
            )
        )

        lines = [
            "# Open LLM Battle Agent 결과 보고서",
            "",
            "## 질문",
            "",
            question,
            "",
            "## 생성 옵션",
            "",
        ]

        generation_options = battle.get(
            "generation_options"
        ) or {}

        lines.extend(
            [
                f"- Max Tokens: "
                f"{generation_options.get('max_tokens', '-')}",
                f"- Temperature: "
                f"{generation_options.get('temperature', '-')}",
                "",
                "## 모델 실행 결과",
                "",
            ]
        )

        battle_results = (
            battle.get("results") or []
        )

        if not battle_results:
            lines.append(
                "실행 결과가 없습니다."
            )
            lines.append("")

        for index, result in enumerate(
            battle_results,
            start=1,
        ):
            model_id = result.get(
                "model_id",
                "-",
            )
            status = result.get(
                "status",
                "-",
            )
            response_time = result.get(
                "response_time_seconds",
                "-",
            )

            lines.extend(
                [
                    f"### {index}. {model_id}",
                    "",
                    f"- 상태: {status}",
                    f"- 응답 시간: {response_time}초",
                    "",
                ]
            )

            if status == "success":
                finish_reason = result.get(
                    "finish_reason",
                    "-",
                )

                usage = result.get(
                    "usage"
                ) or {}

                lines.extend(
                    [
                        f"- 종료 사유: {finish_reason}",
                        (
                            f"- Prompt Tokens: "
                            f"{usage.get('prompt_tokens', '-')}"
                        ),
                        (
                            f"- Completion Tokens: "
                            f"{usage.get('completion_tokens', '-')}"
                        ),
                        (
                            f"- Total Tokens: "
                            f"{usage.get('total_tokens', '-')}"
                        ),
                        "",
                        "#### 답변",
                        "",
                        str(
                            result.get(
                                "answer",
                                "-",
                            )
                        ),
                        "",
                    ]
                )

            else:
                lines.extend(
                    [
                        "#### 오류",
                        "",
                        str(
                            result.get(
                                "error",
                                "알 수 없는 오류",
                            )
                        ),
                        "",
                    ]
                )

        lines.extend(
            [
                "---",
                "",
                "## 비교 평가 순위",
                "",
            ]
        )

        ranking = (
            evaluation.get("ranking") or []
        )

        if ranking:
            lines.extend(
                [
                    "| 순위 | 모델 | 총점 | 관련성 | 완성도 | 명확성 | 속도 | 안정성 |",
                    "|---:|---|---:|---:|---:|---:|---:|---:|",
                ]
            )

            for item in ranking:
                scores = (
                    item.get("scores") or {}
                )

                lines.append(
                    f"| {item.get('rank', '-')} "
                    f"| {item.get('model_id', '-')} "
                    f"| {item.get('total_score', 0)} "
                    f"| {scores.get('relevance', 0)} "
                    f"| {scores.get('completeness', 0)} "
                    f"| {scores.get('clarity', 0)} "
                    f"| {scores.get('speed', 0)} "
                    f"| {scores.get('stability', 0)} |"
                )

        else:
            lines.append(
                "평가 결과가 없습니다."
            )

        lines.extend(
            [
                "",
                "## 모델별 세부 평가",
                "",
            ]
        )

        for item in ranking:
            lines.extend(
                [
                    (
                        f"### {item.get('rank', '-')}위 "
                        f"- {item.get('model_id', '-')}"
                    ),
                    "",
                    (
                        f"- 총점: "
                        f"{item.get('total_score', 0)}점"
                    ),
                    (
                        f"- 응답 시간: "
                        f"{item.get('response_time_seconds', '-')}초"
                    ),
                    (
                        f"- 답변 길이: "
                        f"{item.get('answer_length', 0)}자"
                    ),
                    (
                        "- 일치 키워드: "
                        + (
                            ", ".join(
                                item.get(
                                    "matched_keywords",
                                    [],
                                )
                            )
                            or "없음"
                        )
                    ),
                    "",
                ]
            )

        winner = (
            evaluation.get("winner") or {}
        )

        lines.extend(
            [
                "---",
                "",
                "## 최종 추천 모델",
                "",
                (
                    f"**{winner.get('model_id', '선정 불가')}**"
                ),
                "",
                (
                    f"- 총점: "
                    f"{winner.get('total_score', '-')}점"
                ),
                (
                    f"- 선정 이유: "
                    f"{winner.get('reason', '-')}"
                ),
                "",
                "## 주의사항",
                "",
                (
                    "> "
                    + str(
                        evaluation.get(
                            "important_notice",
                            (
                                "정답 데이터가 없으므로 "
                                "사실 정확성은 자동 확정할 수 없습니다."
                            ),
                        )
                    )
                ),
                "",
            ]
        )

        markdown_content = "\n".join(
            lines
        )

        output_path.write_text(
            markdown_content,
            encoding="utf-8",
        )

        return (
            "성공: Markdown 보고서가 생성되었습니다.\n"
            f"파일 경로: {output_path.resolve()}"
        )

    except json.JSONDecodeError:
        return (
            "실패: 입력된 결과가 "
            "올바른 JSON 형식이 아닙니다."
        )

    except Exception as error:
        return (
            f"실패: Markdown 생성 오류 - {error}"
        )


CUSTOM_TOOLS = [
    search_openrouter_llms,
    run_openrouter_battle,
    evaluate_llm_responses,
    generate_report_markdown,
]

## 5. 도구 정보 확인

생성된 도구의 메타데이터를 확인합니다.

In [ ]:
# 도구 목록 등록
CUSTOM_TOOLS = [
    search_openrouter_llms,
    run_openrouter_battle,
    evaluate_llm_responses,
    generate_report_markdown,
]

# 도구 정보 확인
for tool in CUSTOM_TOOLS:
    print(f"\n도구명: {tool.name}")
    print(f"설명: {tool.description}")
    print(f"입력값: {tool.args}")
    print("-" * 80)


도구명: search_openrouter_llms
설명: OpenRouter에서 현재 무료로 실행 가능한 오픈 LLM을 검색합니다.
입력값: {'search_query': {'description': '모델명 또는 개발사 검색어입니다. 예: Qwen, Gemma.', 'title': 'Search Query', 'type': 'string'}, 'language': {'default': 'multilingual', 'description': '원하는 지원 언어 조건입니다.', 'title': 'Language', 'type': 'string'}, 'model_count': {'default': 3, 'description': '반환할 모델 수이며 1 이상 5 이하입니다.', 'title': 'Model Count', 'type': 'integer'}}
--------------------------------------------------------------------------------

도구명: run_openrouter_battle
설명: 여러 오픈 LLM에 같은 질문을 전달하여 답변과 실행 성능을 측정합니다.
입력값: {'model_ids': {'description': '비교할 OpenRouter 무료 모델 ID 목록입니다.', 'items': {'type': 'string'}, 'title': 'Model Ids', 'type': 'array'}, 'question': {'description': '모든 모델에 동일하게 전달할 질문입니다.', 'title': 'Question', 'type': 'string'}, 'max_tokens': {'default': 256, 'description': '최대 출력 토큰 수이며 16 이상 1024 이하입니다.', 'title': 'Max Tokens', 'type': 'integer'}, 'temperature': {'default': 0.2, 'description': '답변 무작위성 값이며 0 이상

## 6. 도구 단독 실행 테스트

**TODO: 다양한 입력값으로 도구를 테스트하세요**

테스트 케이스를 최소 3개 이상 작성하세요:
1. 정상 케이스
2. 엣지 케이스 (경계값)
3. 에러 케이스 (잘못된 입력)

In [ ]:
import json

print("=" * 80)
print("테스트 1: 정상 케이스 — 모델 검색")
print("=" * 80)

search_result = search_openrouter_llms.invoke({
    "search_query": "Qwen",
    "language": "Korean",
    "model_count": 2,
})

print(search_result)
print()


# 검색 결과에서 실제 실행 가능한 모델 ID 추출
model_ids = []

try:
    search_data = json.loads(search_result)

    if search_data.get("status") == "success":
        model_ids = [
            model["model_id"]
            for model in search_data.get("models", [])
        ]

except json.JSONDecodeError:
    print("검색 결과를 JSON으로 변환하지 못했습니다.")


if model_ids:
    print("=" * 80)
    print("테스트 2: 정상 케이스 — LLM 배틀 실행")
    print("=" * 80)

    question = (
        "인공지능을 처음 공부하는 학생에게 "
        "머신러닝과 딥러닝의 차이를 쉽게 설명해주세요."
    )

    battle_result = run_openrouter_battle.invoke({
        "model_ids": model_ids,
        "question": question,
        "max_tokens": 256,
        "temperature": 0.2,
    })

    print(battle_result)
    print()


    print("=" * 80)
    print("테스트 3: 정상 케이스 — 모델 응답 비교 평가")
    print("=" * 80)

    evaluation_result = evaluate_llm_responses.invoke({
        "question": question,
        "battle_results_json": battle_result,
    })

    print(evaluation_result)
    print()


    print("=" * 80)
    print("테스트 4: 정상 케이스 — Markdown 보고서 생성")
    print("=" * 80)

    markdown_result = generate_report_markdown.invoke({
        "battle_results_json": battle_result,
        "evaluation_results_json": evaluation_result,
        "report_filename": "open_llm_battle_test.md",
    })

    print(markdown_result)
    print()

else:
    print(
        "실행 가능한 모델을 찾지 못해 "
        "배틀·평가·Markdown 테스트를 건너뜁니다."
    )


print("=" * 80)
print("테스트 5: 엣지 케이스 — 최소 모델 개수")
print("=" * 80)

edge_result = search_openrouter_llms.invoke({
    "search_query": "Gemma",
    "language": "multilingual",
    "model_count": 1,  # 허용되는 최소값
})

print(edge_result)
print()


print("=" * 80)
print("테스트 6: 에러 케이스 — 잘못된 모델 개수")
print("=" * 80)

error_search_result = search_openrouter_llms.invoke({
    "search_query": "Qwen",
    "language": "Korean",
    "model_count": 0,  # 허용 범위: 1~5
})

print(error_search_result)
print()


print("=" * 80)
print("테스트 7: 에러 케이스 — 빈 모델 목록")
print("=" * 80)

error_battle_result = run_openrouter_battle.invoke({
    "model_ids": [],
    "question": "AI란 무엇인가요?",
    "max_tokens": 256,
    "temperature": 0.2,
})

print(error_battle_result)
print()


print("=" * 80)
print("테스트 8: 에러 케이스 — 잘못된 JSON 평가")
print("=" * 80)

error_evaluation_result = evaluate_llm_responses.invoke({
    "question": "AI란 무엇인가요?",
    "battle_results_json": "올바르지 않은 JSON",
})

print(error_evaluation_result)
print()


print("=" * 80)
print("테스트 9: 에러 케이스 — 잘못된 Markdown 입력")
print("=" * 80)

error_markdown_result = generate_report_markdown.invoke({
    "battle_results_json": "잘못된 JSON",
    "evaluation_results_json": "잘못된 JSON",
    "report_filename": "error_test.md",
})

print(error_markdown_result)

테스트 1: 정상 케이스 — 모델 검색
{
  "status": "failed",
  "message": "현재 실행 가능한 조건 일치 모델을 찾지 못했습니다.",
  "search_query": "Qwen"
}

실행 가능한 모델을 찾지 못해 배틀·평가·PDF 테스트를 건너뜁니다.
테스트 5: 엣지 케이스 — 최소 모델 개수
{
  "status": "success",
  "search_query": "Gemma",
  "language": "multilingual",
  "model_count": 1,
  "models": [
    {
      "model_id": "google/gemma-4-26b-a4b-it:free",
      "name": "Google: Gemma 4 26B A4B  (free)",
      "owned_by": "google",
      "language_condition": "multilingual",
      "description": "Gemma 4 26B A4B IT is an instruction-tuned Mixture-of-Experts (MoE) model from Google DeepMind. Despite 25.2B total parameters, only 3.8B activate per token during inference — delivering near-31B quality at...",
      "context_length": 262144,
      "pricing": {
        "prompt": "0",
        "completion": "0"
      },
      "supported_parameters": [
        "include_reasoning",
        "max_tokens",
        "reasoning",
        "response_format",
        "seed",
        "temperature",
        

## 7. 여러 도구 통합 테스트

팀에서 만든 여러 도구를 함께 테스트합니다.

**TODO: 생성한 모든 도구를 리스트로 정리하세요**

In [ ]:
# ============================================================
# 7. 여러 도구 통합 테스트
# ============================================================

# 팀에서 생성한 모든 도구 등록
CUSTOM_TOOLS = [
    search_openrouter_llms,
    run_openrouter_battle,
    evaluate_llm_responses,
    generate_report_markdown,
]

print(f"총 {len(CUSTOM_TOOLS)}개의 도구가 준비되었습니다.\n")

print("=" * 80)

for i, tool in enumerate(CUSTOM_TOOLS, start=1):
    print(f"{i}. {tool.name}")
    print(f"   설명: {tool.description}")
    print(f"   입력값: {tool.args}")
    print()

총 4개의 도구가 준비되었습니다.

1. search_openrouter_llms
   설명: OpenRouter에서 현재 무료로 실행 가능한 오픈 LLM을 검색합니다.
   입력값: {'search_query': {'description': '모델명 또는 개발사 검색어입니다. 예: Qwen, Gemma.', 'title': 'Search Query', 'type': 'string'}, 'language': {'default': 'multilingual', 'description': '원하는 지원 언어 조건입니다.', 'title': 'Language', 'type': 'string'}, 'model_count': {'default': 3, 'description': '반환할 모델 수이며 1 이상 5 이하입니다.', 'title': 'Model Count', 'type': 'integer'}}

2. run_openrouter_battle
   설명: 여러 오픈 LLM에 같은 질문을 전달하여 답변과 실행 성능을 측정합니다.
   입력값: {'model_ids': {'description': '비교할 OpenRouter 무료 모델 ID 목록입니다.', 'items': {'type': 'string'}, 'title': 'Model Ids', 'type': 'array'}, 'question': {'description': '모든 모델에 동일하게 전달할 질문입니다.', 'title': 'Question', 'type': 'string'}, 'max_tokens': {'default': 256, 'description': '최대 출력 토큰 수이며 16 이상 1024 이하입니다.', 'title': 'Max Tokens', 'type': 'integer'}, 'temperature': {'default': 0.2, 'description': '답변 무작위성 값이며 0 이상 2 이하입니다.', 'title': 'Temperature', 'type': 'number'}}

## 프로젝트 체크리스트

**완료한 항목을 확인하세요:**

- [ ] 팀 도메인 선정 및 필요한 도구 정의 완료
- [ ] AI 프롬프트를 사용하여 도구 코드 생성 완료
- [ ] 최소 3개 이상의 도구 생성 완료
- [ ] 각 도구별 단독 실행 테스트 완료
- [ ] 정상/엣지/에러 케이스 테스트 완료
- [ ] 도구 메타데이터 확인 완료

---

## 다음 단계

생성한 도구를 domain-agent에 통합하세요:

1. `../src/domain-agent/tools.py` 파일 열기
2. TODO 주석을 참고하여 생성한 도구 코드 추가
3. `../src/domain-agent/agent.py` 파일 열기
4. TODO 주석을 참고하여 시스템 프롬프트와 도구 리스트 수정
5. LangGraph Studio로 테스트

---

## 참고 자료

- [LangChain Tools 문서](https://python.langchain.com/docs/modules/agents/tools/)
- [LangChain Custom Tools](https://python.langchain.com/docs/modules/agents/tools/custom_tools/)
- [@tool 데코레이터](https://python.langchain.com/docs/modules/agents/tools/custom_tools/#tool-decorator)